In [ ]:
import os
import subprocess
import sys
import glob
import re
import shutil
from itertools import product

# NLGCL Kaggle Runner v28 (Notebook-only Runtime Patch for Similarity Rule)

# Helper to suppress output
def run_silent(cmd):
    if isinstance(cmd, list):
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def update_yaml_params(config_path, params):
    with open(config_path, 'r', encoding='utf-8') as f:
        content = f.read()

    if 'user_inter_num_interval' in content:
        content = re.sub(r'user_inter_num_interval:.*', 'user_inter_num_interval: "[0,inf)"', content)
    else:
        content += '\nuser_inter_num_interval: "[0,inf)"'

    if 'item_inter_num_interval' in content:
        content = re.sub(r'item_inter_num_interval:.*', 'item_inter_num_interval: "[0,inf)"', content)
    else:
        content += '\nitem_inter_num_interval: "[0,inf)"'

    if 'field_separator' in content:
        content = re.sub(r'field_separator:.*', 'field_separator: "\\t"', content)
    else:
        content += '\nfield_separator: "\\t"'

    content = re.sub(r'val_interval:.*', '', content)

    for key, value in params.items():
        line = f"{key}: {value}"
        if re.search(rf'^{re.escape(key)}\s*:', content, flags=re.MULTILINE):
            content = re.sub(rf'^{re.escape(key)}\s*:.*$', line, content, flags=re.MULTILINE)
        else:
            content += f"\n{line}"

    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(content)


def parse_metrics(log_text):
    pattern = (
        r'recall@10\s*:\s*([0-9]*\.?[0-9]+)\s*'
        r'recall@20\s*:\s*([0-9]*\.?[0-9]+)\s*'
        r'recall@50\s*:\s*([0-9]*\.?[0-9]+)\s*'
        r'ndcg@10\s*:\s*([0-9]*\.?[0-9]+)\s*'
        r'ndcg@20\s*:\s*([0-9]*\.?[0-9]+)\s*'
        r'ndcg@50\s*:\s*([0-9]*\.?[0-9]+)'
    )
    matches = re.findall(pattern, log_text, flags=re.IGNORECASE)
    if not matches:
        return None
    r10, r20, r50, n10, n20, n50 = matches[-1]
    return {
        'recall@10': float(r10),
        'recall@20': float(r20),
        'recall@50': float(r50),
        'ndcg@10': float(n10),
        'ndcg@20': float(n20),
        'ndcg@50': float(n50),
    }


def run_train_once(dataset='QB-video'):
    proc = subprocess.run(
        [sys.executable, 'main.py', '--dataset', dataset],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace'
    )
    log_text = proc.stdout or ''
    metrics = parse_metrics(log_text)
    return proc.returncode, metrics, log_text


def patch_nlgcl_similarity_runtime():
    # Notebook-only runtime patch: no manual edits outside this notebook.
    model_path = 'recbole_gnn/model/general_recommender/nlgcl.py'
    yaml_path = 'recbole_gnn/properties/model/NLGCL.yaml'

    if not os.path.exists(model_path):
        print('Skip patch: nlgcl.py not found.')
        return

    with open(model_path, 'r', encoding='utf-8') as f:
        content = f.read()

    updated = content

    # 1) Inject threshold config
    if "self.pos_sim_threshold" not in updated:
        updated = updated.replace(
            "        self.alpha = config['alpha']\n",
            "        self.alpha = config['alpha']\n        self.pos_sim_threshold = config['pos_sim_threshold'] if 'pos_sim_threshold' in config else 0.1\n"
        )

    # 2) Similarity-aware positive rule in CL loss
    old_block = (
        "            cl_u = cl_u + self.InfoNCE(cur_embedding_i[pos_item], ego_embedding_u[user],\n"
        "                                     ego_embedding_u[user]) + 1e-6\n"
        "            cl_i = cl_i + self.InfoNCE(cur_embedding_u[user], ego_embedding_i[pos_item],\n"
        "                                     ego_embedding_i[pos_item]) + 1e-6\n"
    )
    new_block = (
        "            pair_sim = F.cosine_similarity(cur_embedding_u[user], cur_embedding_i[pos_item], dim=1)\n"
        "            pos_mask = pair_sim > self.pos_sim_threshold\n"
        "            if pos_mask.any():\n"
        "                cl_u = cl_u + self.InfoNCE(cur_embedding_i[pos_item][pos_mask], ego_embedding_u[user][pos_mask],\n"
        "                                         ego_embedding_u[user][pos_mask]) + 1e-6\n"
        "                cl_i = cl_i + self.InfoNCE(cur_embedding_u[user][pos_mask], ego_embedding_i[pos_item][pos_mask],\n"
        "                                         ego_embedding_i[pos_item][pos_mask]) + 1e-6\n"
        "            else:\n"
        "                cl_u = cl_u + self.InfoNCE(cur_embedding_i[pos_item], ego_embedding_u[user],\n"
        "                                         ego_embedding_u[user]) + 1e-6\n"
        "                cl_i = cl_i + self.InfoNCE(cur_embedding_u[user], ego_embedding_i[pos_item],\n"
        "                                         ego_embedding_i[pos_item]) + 1e-6\n"
    )
    if old_block in updated:
        updated = updated.replace(old_block, new_block)

    if updated != content:
        with open(model_path, 'w', encoding='utf-8') as f:
            f.write(updated)
        print('Patched nlgcl.py with similarity-aware positive rule.')
    else:
        print('nlgcl.py patch already present or block not matched.')

    # 3) Ensure model yaml has threshold key to avoid missing config key
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r', encoding='utf-8') as f:
            yml = f.read()
        if 'pos_sim_threshold:' not in yml:
            yml = yml.rstrip() + '\npos_sim_threshold: 0.1\n'
            with open(yaml_path, 'w', encoding='utf-8') as f:
                f.write(yml)
            print('Injected pos_sim_threshold into model yaml.')


# 1. Clone (Silent)
if not os.path.exists('main.py'):
    run_silent('git clone https://github.com/yangzeha/NLGCL.git')
    if os.path.exists('NLGCL'):
        os.chdir('NLGCL')

# 2. Patch Code (Silent)
files = glob.glob('recbole/**/*.py', recursive=True) + glob.glob('recbole_gnn/**/*.py', recursive=True)

replacements = [
    (r'np\.float\b', 'float'),
    (r'np\.int\b', 'int'),
    (r'np\.object\b', 'object'),
    (r'np\.bool\b', 'bool'),
    (r'np\.str\b', 'str'),
    (r'np\.long\b', 'int'),
]

for file_path in files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        new_content = content

        if 'def compatibility_settings(self):' in new_content:
            new_content = re.sub(
                r'def compatibility_settings\(self\):.*?(?=\n\s+def|\Z)',
                'def compatibility_settings(self):\n        pass\n\n    ',
                new_content,
                flags=re.DOTALL
            )

        for pattern, replacement in replacements:
            new_content = re.sub(pattern, replacement, new_content)

        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
    except Exception:
        pass

# Runtime similarity patch (critical when source files were reverted)
patch_nlgcl_similarity_runtime()

# 3. Setup Dependencies (Silent execution using subprocess)
setup_cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "numpy>=2.0.0", "pandas>=2.2.2", "--force-reinstall", "--upgrade"],
    [sys.executable, "-m", "pip", "install", "-q", "torch==2.4.0", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"],
    [sys.executable, "-m", "pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", "https://data.pyg.org/whl/torch-2.4.0+cu121.html"]
]
for cmd in setup_cmds:
    run_silent(cmd)

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt>=0.2.7
torch_geometric
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)
run_silent([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 4. Dataset Preprocessing (Similarity-aware positive samples)
preprocess_script = r"""
import os
import pandas as pd

DATASET_DIR = 'dataset/QB-video'
INTER_FILE = os.path.join(DATASET_DIR, 'QB-video.inter')
NEG_FILE = os.path.join(DATASET_DIR, 'QB-video.neg.csv')
RAW_CSV_PATH = 'dataset/QB-video.csv'
POS_SIM_THRESHOLD = 0.1

os.makedirs(DATASET_DIR, exist_ok=True)

try:
    df = pd.read_csv(RAW_CSV_PATH, low_memory=False)
except Exception:
    df = pd.read_csv(RAW_CSV_PATH, sep='\t', low_memory=False)

if 'video_id' in df.columns and 'item_id' not in df.columns:
    df.rename(columns={'video_id': 'item_id'}, inplace=True)

if 'user_id' not in df.columns or 'item_id' not in df.columns:
    raise ValueError('QB-video.csv must contain user_id and item_id columns.')

edge_candidates = ['edge', 'has_edge', 'is_edge', 'connected', 'click']
edge_col = next((c for c in edge_candidates if c in df.columns), None)

if edge_col is not None:
    edge_flag = pd.to_numeric(df[edge_col], errors='coerce').fillna(0) > 0
else:
    behavior_cols = [c for c in ['click', 'follow', 'like', 'share', 'watching_times'] if c in df.columns]
    if behavior_cols:
        edge_flag = (df[behavior_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1) > 0)
    else:
        edge_flag = pd.Series([True] * len(df), index=df.index)

sim_candidates = ['similarity', 'sim', 'sim_score', 'ui_similarity', 'ui_sim', 'user_item_similarity', 'cosine_sim']
sim_col = next((c for c in sim_candidates if c in df.columns), None)

if sim_col is not None:
    sim_score = pd.to_numeric(df[sim_col], errors='coerce').fillna(0.0)
else:
    score = pd.Series(0.0, index=df.index)
    if 'click' in df.columns:
        score += 0.35 * (pd.to_numeric(df['click'], errors='coerce').fillna(0) > 0).astype(float)
    if 'follow' in df.columns:
        score += 0.25 * (pd.to_numeric(df['follow'], errors='coerce').fillna(0) > 0).astype(float)
    if 'like' in df.columns:
        score += 0.20 * (pd.to_numeric(df['like'], errors='coerce').fillna(0) > 0).astype(float)
    if 'share' in df.columns:
        score += 0.10 * (pd.to_numeric(df['share'], errors='coerce').fillna(0) > 0).astype(float)
    if 'watching_times' in df.columns:
        wt = pd.to_numeric(df['watching_times'], errors='coerce').fillna(0)
        wt_max = wt.max()
        if wt_max > 0:
            score += 0.10 * (wt / wt_max)
    sim_score = score.clip(0.0, 1.0)

df['__edge__'] = edge_flag.astype(bool)
df['__sim__'] = sim_score.astype(float)

pos_df = df[(df['__edge__']) & (df['__sim__'] > POS_SIM_THRESHOLD)][['user_id', 'item_id']].copy()
neg_df = df[~((df['__edge__']) & (df['__sim__'] > POS_SIM_THRESHOLD))][['user_id', 'item_id']].copy()

def strict_iterative_5core(data, k=5):
    data = data.copy()
    while True:
        prev_len = len(data)
        item_counts = data.groupby('item_id')['user_id'].count()
        valid_items = item_counts[item_counts >= k].index
        data = data[data['item_id'].isin(valid_items)]

        user_counts = data.groupby('user_id')['item_id'].count()
        valid_users = user_counts[user_counts >= k].index
        data = data[data['user_id'].isin(valid_users)]

        if len(data) == prev_len:
            break
    return data

final_df = strict_iterative_5core(pos_df, k=5)

u_count = final_df['user_id'].nunique()
i_count = final_df['item_id'].nunique()
inter_count = len(final_df)

print('[Final Match Result - Similarity Aware]')
print(f'Positive threshold: sim > {POS_SIM_THRESHOLD}, edge == 1')
print(f'Positives before 5-core: {len(pos_df)}')
print(f'Negatives (for analysis): {len(neg_df)}')
print(f'Users after 5-core: {u_count}')
print(f'Items after 5-core: {i_count}')
print(f'Interactions after 5-core: {inter_count}')

final_df.columns = ['user_id:token', 'item_id:token']
final_df.to_csv(INTER_FILE, sep='\t', index=False)
neg_df.to_csv(NEG_FILE, index=False)
print(f'Saved positive interactions to {INTER_FILE}')
print(f'Saved negative pairs to {NEG_FILE}')
"""

with open('preprocess_dataset.py', 'w', encoding='utf-8') as f:
    f.write(preprocess_script)

subprocess.check_call([sys.executable, 'preprocess_dataset.py'])

# 5. Inject baseline config aligned with your best pre-sim setting
config_path = 'properties/QB-video.yaml'
base_params = {
    'n_layers': '2',
    'embedding_size': '64',
    'reg_weight': '0.0001',
    'cl_temp': '0.1',
    'alpha': '0.6',
    'cl_reg': '0.00001',
    'show_progress': 'False',
    'epochs': '500',
    'stopping_step': '500',
    'pos_sim_threshold': '0.1'
}
update_yaml_params(config_path, base_params)
print('Updated properties/QB-video.yaml with similarity-aware baseline params.')

# 6. Optional quick tuning around current best area
ENABLE_PARAM_SEARCH = False

if ENABLE_PARAM_SEARCH:
    search_space = {
        'n_layers': ['2', '3'],
        'cl_temp': ['0.08', '0.1', '0.15'],
        'cl_reg': ['0.000005', '0.00001', '0.00002'],
        'stopping_step': ['500']
    }

    common_params = {
        'embedding_size': '64',
        'reg_weight': '0.0001',
        'alpha': '0.6',
        'show_progress': 'False',
        'epochs': '500',
        'pos_sim_threshold': '0.1'
    }

    keys = ['n_layers', 'cl_temp', 'cl_reg', 'stopping_step']
    all_trials = list(product(*(search_space[k] for k in keys)))
    print(f'Starting quick search with {len(all_trials)} trials...')

    best = None
    for idx, vals in enumerate(all_trials, start=1):
        trial_params = dict(zip(keys, vals))
        merged = {**common_params, **trial_params}
        update_yaml_params(config_path, merged)

        if os.path.exists('saved'):
            shutil.rmtree('saved')
        if os.path.exists('dataset_cache'):
            shutil.rmtree('dataset_cache')

        code, metrics, logs = run_train_once(dataset='QB-video')
        if code != 0 or metrics is None:
            print(f'Trial {idx}/{len(all_trials)} failed or no metrics parsed.')
            continue

        score = metrics['ndcg@20'] + 0.3 * metrics['recall@20']
        print(
            f"Trial {idx}/{len(all_trials)} | params={trial_params} | "
            f"R@20={metrics['recall@20']:.4f} N@20={metrics['ndcg@20']:.4f} score={score:.4f}"
        )

        record = {'params': merged, 'metrics': metrics, 'score': score}
        if best is None or record['score'] > best['score']:
            best = record

    if best is None:
        raise RuntimeError('No successful trial found in quick search.')

    update_yaml_params(config_path, best['params'])
    print('\n[Best Params Found]')
    print(best['params'])
    print(best['metrics'])

# 7. Final training
if os.path.exists('saved'):
    shutil.rmtree('saved')
if os.path.exists('dataset_cache'):
    shutil.rmtree('dataset_cache')

print('Starting Training (similarity-aware positives)...')
ret_code, final_metrics, final_log = run_train_once(dataset='QB-video')
if ret_code != 0:
    raise RuntimeError('Training failed. Check final_log for details.')

if final_metrics is None:
    print('Training finished, but metric line was not parsed. Last 1000 log chars:')
    print(final_log[-1000:])
else:
    print('[Final Metrics]')
    print(
        f"recall@10: {final_metrics['recall@10']:.4f} "
        f"recall@20: {final_metrics['recall@20']:.4f} "
        f"recall@50: {final_metrics['recall@50']:.4f} "
        f"ndcg@10: {final_metrics['ndcg@10']:.4f} "
        f"ndcg@20: {final_metrics['ndcg@20']:.4f} "
        f"ndcg@50: {final_metrics['ndcg@50']:.4f}"
    )